# Notebook 1: Data Check and Cleaning
**Project:** Human-Centered AI Pricing Optimization System for Small Hospitality Businesses in Sri Lanka

What this notebook does:
1. Loads your 3 datasets
2. Checks data quality (missing values, duplicates, estimated values, outliers)
3. Cleans them and saves clean files for the next notebooks

**Before you run:** Put these 3 files in a Google Drive folder called `pricing_project/data/`:
- `sri_lanka_hospitality_pricing_dataset.csv` (main training data, LKR)
- `sri_lanka_tourist_arrival_2023-2025.csv` (extra data, USD)
- `market_data.csv` (scraped competitor prices)

## Step 1: Set up and load the data

In [ ]:
import os
import numpy as np
import pandas as pd

# Works in Google Colab (Drive) or on your own computer
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pricing_project'
except ImportError:
    BASE = os.environ.get('PROJECT_DIR', '.')

DATA = f'{BASE}/data'
CLEAN = f'{BASE}/data/clean'
os.makedirs(CLEAN, exist_ok=True)

main = pd.read_csv(f'{DATA}/sri_lanka_hospitality_pricing_dataset.csv', parse_dates=['Date'])
extra = pd.read_csv(f'{DATA}/sri_lanka_tourist_arrival_2023-2025.csv', parse_dates=['Date'])
market = pd.read_csv(f'{DATA}/market_data.csv')

print('Main dataset  :', main.shape)
print('Extra dataset :', extra.shape)
print('Market data   :', market.shape)
main.head()

## Step 2: Quality check - main dataset (2024-2025, LKR)

In [ ]:
print('Date range:', main.Date.min().date(), 'to', main.Date.max().date())
print('Missing values:', int(main.isna().sum().sum()))
print('Duplicate dates:', int(main.Date.duplicated().sum()))
full_range = pd.date_range(main.Date.min(), main.Date.max(), freq='D')
print('Missing days:', len(full_range.difference(main.Date)))
print()
print(main.Seasonality_Index.value_counts())
main.describe().round(2)

## Step 3: Quality check - extra dataset (2023-2025, USD)

In [ ]:
print('Date range:', extra.Date.min().date(), 'to', extra.Date.max().date())
print('Missing values:', int(extra.isna().sum().sum()))
print('Duplicate dates:', int(extra.Date.duplicated().sum()))
print()
print('Share of ESTIMATED occupancy values by year:')
print(extra.groupby('Year').Occupancy_Is_Estimated.mean().round(2))

> **Important finding:** in the extra dataset, the occupancy for **all of 2025** is *estimated*, not real.
> We will use only 2023-2024 rows as "real" occupancy and mention this in the Limitations section of your thesis.

## Step 4: Quality check - market data (scraped prices)

In [ ]:
print('Platforms :', market.platform.value_counts().to_dict())
print('Districts :', market.district.value_counts().to_dict())
print('Check-in dates:', market.check_in_date.min(), 'to', market.check_in_date.max())
print('Rows marked as anomaly:', int(market.is_anomaly.sum()))
print()
print(market.price_per_night_lkr.describe().round(0))
market[market.is_anomaly == 1][['property_name', 'price_per_night_lkr', 'anomaly_reason']]

## Step 5: Clean the main dataset and add features

In [ ]:
df = main.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)

# Calendar features
df['Day_of_Week'] = df.Date.dt.dayofweek          # 0 = Monday
df['Is_Weekend'] = (df.Day_of_Week >= 5).astype(int)
df['Month'] = df.Date.dt.month

# Seasonality label to number (Off-Peak=0, Shoulder=1, Peak=2)
df['Season_Code'] = df.Seasonality_Index.map({'Off-Peak': 0, 'Shoulder': 1, 'Peak': 2})

# Past-only features (use only yesterday and before, so we never use the future)
df['Arrivals_Lag7'] = df.SLTDA_Tourist_Arrivals.shift(7)
df['Arrivals_Roll7'] = df.SLTDA_Tourist_Arrivals.shift(1).rolling(7).mean()
df['Occupancy_Lag7'] = df.Historical_Occupancy_Rate.shift(7)

df = df.dropna().reset_index(drop=True)
print('Rows after cleaning:', len(df))
df.to_csv(f'{CLEAN}/main_clean.csv', index=False)
df.head()

## Step 6: Clean the extra dataset

In [ ]:
ex = extra.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
ex['Occupancy_Is_Estimated'] = ex.Occupancy_Is_Estimated.astype(int)
ex.to_csv(f'{CLEAN}/extra_clean.csv', index=False)
print('Saved extra_clean.csv with', len(ex), 'rows (2025 occupancy is estimated)')

## Step 7: Clean the market data and make a summary

In [ ]:
mk = market.copy()
mk = mk[mk.is_anomaly == 0]                       # remove the flagged anomaly
mk = mk.drop(columns=['original_price_lkr'])      # column is empty

# extra safety: remove prices far above the normal range (IQR rule)
q1, q3 = mk.price_per_night_lkr.quantile([0.25, 0.75])
upper = q3 + 3 * (q3 - q1)
removed = mk[mk.price_per_night_lkr > upper]
print('Extra outliers removed:', len(removed))
mk = mk[mk.price_per_night_lkr <= upper].reset_index(drop=True)
mk.to_csv(f'{CLEAN}/market_clean.csv', index=False)

summary = (mk.groupby(['district', 'property_type'])
             .price_per_night_lkr.agg(['count', 'median', lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
             .round(0))
summary.columns = ['count', 'median_lkr', 'p25_lkr', 'p75_lkr']
summary = summary.reset_index()
summary.to_csv(f'{CLEAN}/market_summary.csv', index=False)
print('Market rows kept:', len(mk))
summary.head(12)

## Step 8: Quick look at the data (charts)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax[0].plot(df.Date, df.SLTDA_Tourist_Arrivals);   ax[0].set_title('Tourist arrivals (SLTDA)')
ax[1].plot(df.Date, df.Historical_Occupancy_Rate, color='green'); ax[1].set_title('Hotel occupancy (%)')
ax[2].plot(df.Date, df.Competitor_Avg_Price_LKR, color='orange'); ax[2].set_title('Competitor average price (LKR)')
plt.tight_layout(); plt.show()

## Done. Files saved in `data/clean/`
| File | Use |
|---|---|
| `main_clean.csv` | Train the demand model (Notebook 2) |
| `extra_clean.csv` | Extra check of the model on 2023 data |
| `market_clean.csv` | Competitor prices for the web app |
| `market_summary.csv` | Median price by district and property type |

**Next: Notebook 2 - Demand forecasting model (Linear Regression, Random Forest, XGBoost).**